# Lezione 10 - Agenti di IA in Produzione

In questa lezione imparerai **pattern di produzione** per agenti di IA utilizzando il Microsoft Agent Framework (Python). Copriamo:

- **Osservabilità** — aggiunta di misurazione dei tempi e di logging alle interazioni degli agenti
- **Valutazione** — utilizzo di un agente valutatore per assegnare un punteggio alla qualità delle risposte
- **Gestione dei costi** — strategie per l'ottimizzazione dei token e la selezione del modello

Lo scenario è un **agente di viaggio** che aiuta gli utenti a pianificare i loro viaggi, con monitoraggio e valutazione integrati.


## Configurazione


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Considerazioni per la produzione

Trasferire agenti AI dai prototipi alla produzione richiede un'attenzione accurata a tre pilastri:

1. **Osservabilità** — Hai bisogno di visibilità su ciò che l'agente sta facendo, quanto tempo impiega e quali strumenti invoca. Senza tracciamento e registrazione dei log, il debug dei problemi in produzione è quasi impossibile.

2. **Valutazione** — Controlli di qualità automatizzati garantiscono che le risposte dell'agente rimangano accurate, complete e utili nel tempo. Un agente valutatore può assegnare un punteggio alle risposte in base a criteri definiti.

3. **Gestione dei costi** — L'utilizzo dei token incide direttamente sui costi. Strategie come l'ottimizzazione dei prompt, la selezione del modello e la memorizzazione nella cache aiutano a mantenere le spese sotto controllo senza compromettere la qualità.


## Creazione di un agente osservabile

Definiamo gli strumenti di viaggio e avvolgiamo la chiamata all'agente con una misurazione del tempo in modo da poter monitorare la latenza. In produzione integreresti OpenTelemetry o un backend di tracing simile.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Modelli di valutazione

Un modello comune in produzione è utilizzare un secondo agente come **valutatore**. Il valutatore assegna un punteggio alla risposta dell'agente primario secondo criteri predefiniti come completezza, accuratezza e utilità.

Questo consente:
- Controlli di qualità automatizzati prima che le risposte raggiungano gli utenti
- Rilevamento delle regressioni quando prompt o modelli cambiano
- Monitoraggio continuo delle prestazioni dell'agente nel tempo


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie per la gestione dei costi

Controllare i costi è fondamentale per gli agenti AI in produzione. Ecco le strategie chiave:

| Strategia | Descrizione |
|---|---|
| **Prompt optimization** | Mantieni le istruzioni di sistema concise. Rimuovi il contesto ridondante per ridurre i token di input. |
| **Model selection** | Usa modelli più piccoli e economici (ad es. GPT-4o-mini) per attività semplici come classificazione o estrazione, e riserva i modelli più grandi per il ragionamento complesso. |
| **Caching** | Memorizza nella cache i risultati degli strumenti e le query frequenti per evitare chiamate API ridondanti. |
| **Token budgets** | Imposta limiti `max_tokens` per prevenire risposte inaspettatamente lunghe. |
| **Batching** | Raggruppa più richieste degli utenti in una singola chiamata API quando possibile. |

In pratica, un approccio a livelli funziona bene: indirizza le richieste semplici a un modello veloce e poco costoso e scala solo le query complesse verso uno più capace.


## Riepilogo

In questa lezione hai imparato a:

1. **Aggiungere osservabilità** alle interazioni degli agenti con la misurazione dei tempi e la registrazione dei log, ponendo le basi per il tracciamento e il monitoraggio.
2. **Valutare le risposte dell'agente** automaticamente utilizzando un agente valutatore che assegna un punteggio a completezza, accuratezza e utilità.
3. **Gestire i costi** tramite l'ottimizzazione dei prompt, la selezione del modello, il caching e i budget di token.

Questi pattern di produzione aiutano a garantire che i tuoi agenti AI siano affidabili, misurabili ed efficienti in termini di costi su larga scala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Dichiarazione di non responsabilità:
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica basato su intelligenza artificiale [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per l'accuratezza, si prega di notare che le traduzioni automatiche possono contenere errori o imprecisioni. Il documento originale nella sua lingua d'origine deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un traduttore umano. Non siamo responsabili per eventuali incomprensioni o interpretazioni errate derivanti dall'uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
